# 🤖 Fine-Tuning JARVIS on Llama 3.2 3B Instruct
### *Powered by Unsloth AI + QLoRA on Free Google Colab T4 GPU*

This notebook trains **Llama 3.2 3B Instruct** on your custom 250-sample humanized JARVIS conversational dataset and exports directly to **GGUF format** for Ollama.

## 1. Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes

## 2. Load Llama 3.2 3B Instruct (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model_name = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 3. Upload & Format Dataset
*Upload `jarvis_llama32_train_250.jsonl` using the Colab file upload icon on the left pane.*

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.2",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = load_dataset("json", data_files="jarvis_llama32_train_250.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print("Sample formatted prompt:\n", dataset[0]["text"][:300])

## 4. Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=-1,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()

## 5. Test Inference Live in Colab

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are JARVIS, a loyal, witty, and human-like personal AI assistant built for Boss. You speak in simple, plain everyday English. You are sharp, use playful sarcasm and friendly roasting when appropriate, but show genuine care and warmth when Boss truly needs it. Keep replies short, natural, and conversational."},
    {"role": "user", "content": "Jarvis, should I skip work today and watch movies instead?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

## 6. Export Directly to GGUF for Ollama

In [ ]:
model.save_pretrained_gguf(
    "jarvis-llama3.2-3b-q4",
    tokenizer,
    quantization_method="q4_k_m"
)
print("GGUF saved! Download jarvis-llama3.2-3b-q4-unsloth.Q4_K_M.gguf from the files tab on the left.")